In [ ]:
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
import gradio as gr

In [ ]:
# Connect to Ollama - no API keys needed
ollama = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

MODEL = "llama3.2"

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

WIKI_PAGES = {
    "quantum":           "https://en.wikipedia.org/wiki/Quantum",
    "quantum_mechanics":  "https://en.wikipedia.org/wiki/Quantum_mechanics",
    "quantum_computing":  "https://en.wikipedia.org/wiki/Quantum_computing",
    "quantum_entanglement": "https://en.wikipedia.org/wiki/Quantum_entanglement",
    "quantum_field_theory": "https://en.wikipedia.org/wiki/Quantum_field_theory",
    "planck_constant":    "https://en.wikipedia.org/wiki/Planck_constant",
    "wave_function":      "https://en.wikipedia.org/wiki/Wave_function",
}

def fetch_page(url, max_chars=1500):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(resp.content, "html.parser")
        title = soup.title.string.strip() if soup.title else "No title"
        if soup.body:
            for tag in soup.body(["script", "style", "img", "input", "nav", "footer"]):
                tag.decompose()
            text = soup.body.get_text(separator="\n", strip=True)
        else:
            text = ""
        return f"Page: {title}\nURL: {url}\n\n{text}"[:max_chars]
    except Exception as e:
        return f"Could not fetch {url}: {e}"

def fetch_wiki_context():
    parts = []
    for name, url in WIKI_PAGES.items():
        print(f"Fetching {name} page...")
        parts.append(fetch_page(url))
    return "\n\n---\n\n".join(parts)

print("Loading Wikipedia Quantum content...")
wiki_context = fetch_wiki_context()
print(f"Done\! Fetched {len(wiki_context)} characters of content.")

In [ ]:
system_message = """You are a quantum physics expert assistant.
You explain quantum concepts clearly, from beginner to advanced level.
Your answers are grounded in the Wikipedia content provided.
Be concise, accurate, and engaging. Respond in markdown without code blocks."""

In [ ]:
def stream_ollama(question, model_name):
    user_prompt = "Based on the following Wikipedia content about Quantum, answer this question:\n\n"
    user_prompt += f"Question: {question}\n\n"
    user_prompt += f"Wikipedia content:\n{wiki_context}"

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt}
    ]

    stream = ollama.chat.completions.create(
        model=model_name,
        messages=messages,
        stream=True
    )

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
question_input = gr.Textbox(
    label="Your question about Quantum:",
    info="Ask anything about quantum physics, mechanics, computing, or entanglement",
    lines=3
)
model_selector = gr.Dropdown(
    ["llama3.2"],
    label="Ollama model",
    value="llama3.2"
)
answer_output = gr.Markdown(label="Answer:")

view = gr.Interface(
    fn=stream_ollama,
    title="Quantum Expert (powered by Ollama + Wikipedia)",
    description="Ask questions about quantum physics — answers grounded in Wikipedia content.",
    inputs=[question_input, model_selector],
    outputs=[answer_output],
    examples=[
        ["what is quantum mechanics", "llama3.2"],
        ["what is quantum entanglement", "llama3.2"],
        ["what is quantum computing", "llama3.2"],
        ["what is the Planck constant", "llama3.2"],
        ["what is a wave function", "llama3.2"],
        ["what is quantum field theory", "llama3.2"],
    ],
    flagging_mode="never"
)
view.launch(inbrowser=True)